# TopK clients

Four clients, one backend, one set of collections. The question is what each client costs
on top of the engine -- not how the engine performs, which is `bench.ipynb`.

| | proto/gRPC | pgwire | ES HTTP |
|---|---|---|---|
| **rust** | `topk-rs` | - | - |
| **python** | `topk-py` | `topk-sql` | `topk-es` |
| **js** | `topk-js` *(later)* | - | - |

`topk-rs` is the floor. The harness is itself Rust, so a Rust client adds no
client-language cost: it is the protocol and nothing else. `topk-py` speaks the *same*
protocol and pays PyO3 plus Python on top of it. So the two readings separate cleanly:

- `topk-rs` -> `topk-py` is **client language**
- `topk-py` -> `topk-es` is **protocol**

Everything below is plotted in absolute units, side by side. Absolutes drift with
whatever staging was doing that hour -- roughly 20% between days -- so read the *spacing*
between clients within a chart, not the height of any single point across charts. The
provider loop is the innermost loop in the sweep for exactly this reason: every client is
measured minutes apart under the same conditions.

Preflight gates each sweep and its manifest sits beside the parquet: endpoint identity,
document counts, and a cross-client parity check that all four return the same documents
for the same query. A sweep without a passing manifest is not plotted.


In [1]:
import os, glob
import polars as pl
import plotly.express as px

# The harness emits the python proto provider as "topk"; name it for what it is.
RENAME   = {"topk": "topk-py"}
ORDER    = ["topk-rs", "topk-py", "topk-sql", "topk-es", "topk-js"]
PREFERRED_BASELINE = "topk-rs"      # the real floor: Rust client, Rust harness

def repo_root():
    d = os.path.abspath(os.getcwd())
    while d != "/":
        if os.path.isfile(os.path.join(d, "local.py")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("could not locate repo root (no local.py above cwd)")

ROOT = repo_root()

# One sweep, loaded whole: query parquet at the top level, ingest under ingest/<size>/b<N>/,
# manifest.json beside them. Nothing is pooled across sweeps -- earlier ones ran against an
# es-proxy that ignored `_source_includes`, so every topk-es hit carried its 768-float
# embedding. Those live in stash/archive-2026-08-02/ and are deliberately unreachable here.
SWEEP      = os.environ.get("BENCH_SWEEP", "clients3-20260803T2330Z")
QUERY_DIR  = os.path.join(ROOT, "results", SWEEP)
BATCH_DIRS = [os.path.join(ROOT, "results", SWEEP, "ingest")]

QUERY_FILES = sorted(glob.glob(os.path.join(QUERY_DIR, "*.parquet")))
# Batch sweep: results/batch-*/b<N>/*.parquet, one directory per batch size.
# ingest/<size>/b<N>/*.parquet (campaign 2) or b<N>/*.parquet (the earlier flat sweep)
BATCH_FILES = []
for _d in BATCH_DIRS:
    BATCH_FILES += sorted(glob.glob(os.path.join(_d, "**", "b*", "*.parquet"), recursive=True))
    BATCH_FILES += sorted(glob.glob(os.path.join(_d, "b*", "*.parquet")))
BATCH_FILES = sorted(set(BATCH_FILES))

def load(files):
    """Concatenate a sweep, minus its warmup passes.

    Every mode warms up before measuring, and the harness tags those rows `warmup=true`
    while giving them the same concurrency (1) and top_k (10) as real points. Pooled,
    they *dominated* the c=1 and k=10 columns -- and they are cold connections, which is
    not a property of a client worth charting.
    """
    if not files:
        return None
    d = pl.concat([pl.read_parquet(f) for f in files])
    # Query parquet has the column; ingest parquet does not -- the write path never warms up.
    if "warmup" in d.columns:
        d = d.filter(pl.col("warmup") != "true")
    return d.with_columns(pl.col("provider").replace(RENAME))

df, ing = load(QUERY_FILES), load(BATCH_FILES)

present = set()
for frame in (df, ing):
    if frame is not None:
        present |= set(frame["provider"].unique().to_list())
BASELINE = PREFERRED_BASELINE if PREFERRED_BASELINE in present else "topk-py"
if BASELINE != PREFERRED_BASELINE:
    print(f"   note: {PREFERRED_BASELINE} not in this dataset -- topk-py is the fastest client here,"
          f" but it still pays PyO3 + Python, so it is not the protocol floor.")
SIZES = [s for s in ["100k", "1m", "10m"] if df is not None and s in df["size"].unique().to_list()]

print(f"query {len(QUERY_FILES)} files, batch-sweep {len(BATCH_FILES)} files, sizes {SIZES}")
if df is not None:
    with pl.Config(tbl_rows=-1):
        print(df.group_by(["provider", "mode", "size"])
                .agg(pl.col("run_id").n_unique().alias("runs")).sort(["mode", "size", "provider"]))

query 40 files, batch-sweep 36 files, sizes ['100k', '1m']


shape: (16, 4)
┌──────────┬────────┬──────┬──────┐
│ provider ┆ mode   ┆ size ┆ runs │
│ ---      ┆ ---    ┆ ---  ┆ ---  │
│ str      ┆ str    ┆ str  ┆ u32  │
╞══════════╪════════╪══════╪══════╡
│ topk-es  ┆ ksweep ┆ 100k ┆ 8    │
│ topk-py  ┆ ksweep ┆ 100k ┆ 8    │
│ topk-rs  ┆ ksweep ┆ 100k ┆ 8    │
│ topk-sql ┆ ksweep ┆ 100k ┆ 8    │
│ topk-es  ┆ ksweep ┆ 1m   ┆ 8    │
│ topk-py  ┆ ksweep ┆ 1m   ┆ 8    │
│ topk-rs  ┆ ksweep ┆ 1m   ┆ 8    │
│ topk-sql ┆ ksweep ┆ 1m   ┆ 8    │
│ topk-es  ┆ qps    ┆ 100k ┆ 12   │
│ topk-py  ┆ qps    ┆ 100k ┆ 12   │
│ topk-rs  ┆ qps    ┆ 100k ┆ 12   │
│ topk-sql ┆ qps    ┆ 100k ┆ 12   │
│ topk-es  ┆ qps    ┆ 1m   ┆ 12   │
│ topk-py  ┆ qps    ┆ 1m   ┆ 12   │
│ topk-rs  ┆ qps    ┆ 1m   ┆ 12   │
│ topk-sql ┆ qps    ┆ 1m   ┆ 12   │
└──────────┴────────┴──────┴──────┘


In [2]:
# One value per (group, run_id) first, so a run is the unit of repetition and the
# whiskers below are run-to-run spread.

def per_run(frame, keys, kind):
    if kind == "latency":
        return (frame.filter(pl.col("metric") == "bench.query.latency_ms")
                     .group_by(keys + ["run_id"]).agg(pl.col("value").quantile(0.99).alias("value")))
    if kind == "qps":
        return (frame.filter(pl.col("metric") == "bench.query.latency_ms")
                     .group_by(keys + ["run_id"])
                     .agg([pl.col("ts").min().alias("s"), pl.col("ts").max().alias("e"),
                           pl.len().alias("n")])
                     .with_columns((pl.col("n") /
                         ((pl.col("e") - pl.col("s")).dt.total_seconds())).alias("value"))
                     .select(keys + ["run_id", "value"]))
    if kind == "recall":
        return (frame.filter(pl.col("metric") == "bench.query.recall")
                     .group_by(keys + ["run_id"]).agg(pl.col("value").mean().alias("value")))

def agg(frame, keys, kind):
    return (per_run(frame, keys, kind).group_by(keys).agg([
        pl.col("value").mean().alias("value"),
        pl.col("value").min().alias("lo"),
        pl.col("value").max().alias("hi")]))

def vs_baseline(frame, keys, kind, pct=False):
    """Ratio to BASELINE within the same cell. pct=True -> % of baseline (QPS)."""
    a = agg(frame, keys, kind)
    other = [k for k in keys if k != "provider"]
    base = a.filter(pl.col("provider") == BASELINE).select(other + [pl.col("value").alias("b")])
    j = a.join(base, on=other, how="inner")
    m = 100 if pct else 1
    j = j.with_columns([(pl.col(c) / pl.col("b") * m).alias(c) for c in ("value", "lo", "hi")])
    # plotly error bars are offsets from the bar height, over the same runs as the mean
    return (j.with_columns([(pl.col("hi") - pl.col("value")).alias("err_plus"),
                            (pl.col("value") - pl.col("lo")).alias("err_minus"),
                            pl.col("provider").cast(pl.Enum(ORDER))])
             .sort(other + ["provider"]))

def bars(frame, x, y, title, ylab, ref=None, **kw):
    fig = px.bar(frame.to_pandas(), x=x, y=y, color="provider", barmode="group",
                 error_y="err_plus", error_y_minus="err_minus",
                 category_orders={"size": SIZES, "provider": ORDER},
                 labels={y: ylab}, title=title, **kw)
    if ref is not None:
        fig.add_hline(y=ref, line_dash="dot")
    return fig

unf = (df.filter((pl.col("mode") == "qps") & (pl.col("int_filter") == "")
                 & (pl.col("keyword_filter") == "")) if df is not None else None)

## Latency and throughput

Both are plotted **against concurrency**, never aggregated across it: a p99 taken over a
mixture of c=1 and c=8 is not a percentile of anything.

Warmup passes are dropped at load. They carry `concurrency=1`, so pooling them made the
c=1 point mostly cold-connection data.

In [3]:
if unf is not None and len(unf):
    lat = (agg(unf, ["provider", "size", "concurrency"], "latency")
             .with_columns([(pl.col("hi") - pl.col("value")).alias("err_plus"),
                            (pl.col("value") - pl.col("lo")).alias("err_minus"),
                            pl.col("concurrency").cast(pl.Int32),
                            pl.col("provider").cast(pl.Enum(ORDER))])
             .sort("concurrency"))
    fig = px.line(lat.to_pandas(), x="concurrency", y="value", color="provider",
                  facet_col="size", markers=True,
                  error_y="err_plus", error_y_minus="err_minus",
                  category_orders={"size": SIZES, "provider": ORDER},
                  labels={"value": "p99 ms"}, title="p99 latency vs concurrency")
    fig.show()

## Throughput under concurrency

In [4]:
if unf is not None and len(unf):
    r = (agg(unf, ["provider", "size", "concurrency"], "qps")
           .with_columns([(pl.col("hi") - pl.col("value")).alias("err_plus"),
                          (pl.col("value") - pl.col("lo")).alias("err_minus"),
                          pl.col("concurrency").cast(pl.Int32),
                          pl.col("provider").cast(pl.Enum(ORDER))])
           .sort("concurrency"))
    fig = px.line(r.to_pandas(), x="concurrency", y="value", color="provider",
                  facet_col="size", markers=True,
                  error_y="err_plus", error_y_minus="err_minus",
                  category_orders={"size": SIZES, "provider": ORDER},
                  labels={"value": "queries / second"}, title="Throughput vs concurrency")
    fig.show()

## Result-set size

The bytes-OUT axis. Response size is `k` x bytes-per-hit, so this separates a client's
per-byte cost from its per-request cost -- every other mode pins `top_k=10`, which is a
single point in payload space.

In [5]:
ks = df.filter(pl.col("mode") == "ksweep") if df is not None else None
if ks is not None and len(ks):
    a = (agg(ks, ["provider", "size", "top_k"], "latency")
           .with_columns([(pl.col("hi") - pl.col("value")).alias("err_plus"),
                          (pl.col("value") - pl.col("lo")).alias("err_minus"),
                          pl.col("top_k").cast(pl.Int32),
                          pl.col("provider").cast(pl.Enum(ORDER))])
           .sort("top_k"))
    fig = px.line(a.to_pandas(), x="top_k", y="value", color="provider",
                  facet_col="size", markers=True, log_x=True,
                  error_y="err_plus", error_y_minus="err_minus",
                  category_orders={"size": SIZES, "provider": ORDER},
                  labels={"value": "p99 ms", "top_k": "k (log)"},
                  title="p99 latency vs result-set size")
    fig.show()
else:
    print("no ksweep data yet")

## Recall parity

Same collections, so these must coincide.

In [6]:
rec = df.filter(pl.col("metric") == "bench.query.recall") if df is not None else None
if rec is not None and len(rec):
    a = agg(rec, ["provider", "size"], "recall").with_columns([
        (pl.col("hi") - pl.col("value")).alias("err_plus"),
        (pl.col("value") - pl.col("lo")).alias("err_minus"),
        pl.col("provider").cast(pl.Enum(ORDER))])
    fig = bars(a.sort("size"), "size", "value", "Recall by client", "recall")
    fig.update_yaxes(range=[a["lo"].min() - 0.01, a["hi"].max() + 0.01])
    fig.show()

# A dropped query never contributes a latency sample, so a mismatch means the latency
# distribution is missing exactly the slow requests.
if df is not None:
    oks  = (df.filter(pl.col("metric") == "bench.query.oks")
              .group_by(["provider", "size"]).agg(pl.col("value").sum().alias("oks")))
    lats = (df.filter(pl.col("metric") == "bench.query.latency_ms")
              .group_by(["provider", "size"]).agg(pl.len().alias("samples")))
    bad = (oks.join(lats, on=["provider", "size"])
              .filter(pl.col("oks") != pl.col("samples")))
    print("oks == latency samples: OK" if not len(bad) else f"!! MISMATCH\n{bad}")

oks == latency samples: OK


## Write throughput vs batch size

Batch size is now the **only** knob controlling wire requests: the ES provider used to
re-split every batch on an internal `MAX_BULK_BYTES`, so the harness asking for 2000
documents quietly became ~15 HTTP requests. That constant is gone -- one logical batch
is one request -- and the shim's body limit was raised from axum's 2 MiB default to
64 MiB.

Each client should show an optimum: too small and per-request cost dominates, too large
and you pay for buffering. Where a curve simply stops, the client hit the body limit.

Throughput here is documents divided by *elapsed* time -- last completion minus first
start. The obvious alternative, the spread of completion timestamps, breaks down at the
right-hand end of this very chart: at batch 12000 there are 9 requests and 8 workers, so
they all finish together and the spread tends to zero regardless of how long they took.

In [7]:
import re

def per_file(f):
    """One row per parquet, i.e. per (provider, batch, run)."""
    batch = int(re.search(r"/b(\d+)/", f).group(1))
    d = pl.read_parquet(f)
    lat = d.filter(pl.col("metric") == "bench.ingest.latency_ms")
    if not len(lat):
        return None
    # Elapsed = last completion - first start. Using the spread of completions instead
    # silently inflates throughput once the request count approaches the concurrency,
    # because then every request finishes at roughly the same moment.
    lat = lat.with_columns((pl.col("ts") - pl.duration(milliseconds=pl.col("value"))).alias("start"))
    wall = (lat["ts"].max() - lat["start"].min()).total_seconds()
    if not wall:
        return None
    g = lambda m: d.filter(pl.col("metric") == m)["value"].sum()
    docs, byts, reqs = g("bench.ingest.upserted_docs"), g("bench.ingest.upserted_bytes"), g("bench.ingest.requests")
    return {"provider": RENAME.get(d["provider"][0], d["provider"][0]),
            "batch": batch, "size": d["size"][0], "wall": wall,
            "docs": docs, "bytes": byts,
            "docs_s": docs / wall,
            "MB_per_req": byts / reqs / 1e6 if reqs else 0,
            "reqs": reqs,
            "wire": g("bench.ingest.wire_bytes")}

if BATCH_FILES:
    raw = pl.DataFrame([r for r in (per_file(f) for f in BATCH_FILES) if r])

    # Mean across repeat runs, with the run-to-run range as whiskers -- the same
    # treatment the read charts give their repeats.
    b = (raw.group_by(["provider", "batch", "size"])
            .agg([pl.col("docs_s").mean().alias("docs_s"),
                  pl.col("docs_s").min().alias("lo"), pl.col("docs_s").max().alias("hi"),
                  pl.col("MB_per_req").mean().alias("MB_per_req"),
                  pl.col("reqs").mean().alias("reqs"),
                  pl.len().alias("runs")])
            .with_columns([(pl.col("hi") - pl.col("docs_s")).alias("err_plus"),
                           (pl.col("docs_s") - pl.col("lo")).alias("err_minus"),
                           pl.col("provider").cast(pl.Enum(ORDER))])
            .sort(["size", "provider", "batch"]))

    fig = px.line(b.to_pandas(), x="batch", y="docs_s", color="provider", markers=True,
                  facet_col="size", category_orders={"provider": ORDER, "size": SIZES},
                  log_x=True, error_y="err_plus", error_y_minus="err_minus",
                  labels={"docs_s": "documents / second", "batch": "batch size (docs, log)"},
                  title="Ingest throughput vs batch size")
    fig.show()

    # Where each client peaks, and where it stops. A curve that ends before the others
    # ran out of body limit, not out of headroom -- so print the largest batch that
    # actually completed rather than letting the chart imply the sweep covered everyone.
    for prov in b["provider"].unique().to_list():
        s_ = b.filter(pl.col("provider") == prov).sort("docs_s", descending=True)
        top, reached = s_.head(1), b.filter(pl.col("provider") == prov)["batch"].max()
        print(f"  {str(prov):9} peak {top['docs_s'][0]:8,.0f} docs/s at batch={top['batch'][0]:<6} "
              f"({top['MB_per_req'][0]:.1f} MB/request), largest batch completed = {reached}")


  topk-rs   peak   16,188 docs/s at batch=2000   (6.8 MB/request), largest batch completed = 12000
  topk-py   peak   12,224 docs/s at batch=4000   (13.5 MB/request), largest batch completed = 12000
  topk-sql  peak    1,488 docs/s at batch=500    (1.7 MB/request), largest batch completed = 12000
  topk-es   peak    5,958 docs/s at batch=2000   (6.8 MB/request), largest batch completed = 4000


## Effective throughput — goodput vs bytes moved

`bench.ingest.upserted_bytes` is `doc.approx_size()` summed over the **parsed**
documents, so it is protocol-independent: every client reports the same value for the
same documents. That makes it the right numerator for *useful data delivered* —
**goodput** — as distinct from bytes actually put on the wire.

The gap between them is the protocol's tax:

```
goodput      = upserted_bytes / wall          (measured, protocol-independent)
wire rate    = docs/s x wire_bytes_per_doc    (per-client constant, below)
efficiency   = goodput / wire rate = 1 / inflation
```

`wire_bytes_per_doc` is a calibration constant, not a measurement from the harness:

| client | bytes/doc | how |
|---|---|---|
| `topk-py` | ~3.4 KiB | 768 f32 = 3072 B + text + filters, binary proto — **computed**, close to `approx_size` |
| `topk-es` | 16.22 KiB | orjson ndjson of exactly the fields the provider sends — **measured over 500 real documents** |

The ES figure is the trustworthy one: it predicts the observed batch ceiling (63.4 MiB at
batch=4000 passes, 126.7 MiB at 8000 fails against the 64 MiB limit). The native figure is
computed rather than measured and should be treated as approximate — it is not derived
from anything the harness records.

In [8]:
# topk-rs and topk-py encode identical protobuf, so they share a constant. topk-sql has
# none and is therefore absent from this chart rather than plotted on a guess.
WIRE_KIB_PER_DOC = {"topk-py": 3.4, "topk-rs": 3.4, "topk-es": 16.22}

if BATCH_FILES:
    g = []
    for r in raw.iter_rows(named=True):
        # Measured if the provider reports wire bytes, otherwise the calibration constant.
        if r["wire"]:
            wire_total, src = r["wire"], "measured"
        else:
            kib = WIRE_KIB_PER_DOC.get(r["provider"])
            if kib is None:
                continue
            wire_total, src = r["docs"] * kib * 1024, "constant"
        g.append({"provider": r["provider"], "batch": r["batch"], "size": r["size"], "src": src,
                  "goodput": r["bytes"] / r["wall"] / 1e6,
                  "wire": wire_total / r["wall"] / 1e6,
                  "efficiency": r["bytes"] / wire_total})

    gd = (pl.DataFrame(g)
            .group_by(["provider", "batch", "size", "src"])
            .agg([pl.col("goodput").mean(), pl.col("wire").mean(), pl.col("efficiency").mean()])
            .with_columns(pl.col("provider").cast(pl.Enum(ORDER)))
            .sort(["size", "provider", "batch"]))

    m = gd.unpivot(index=["provider", "batch", "size"], on=["goodput", "wire"],
                   variable_name="kind", value_name="MB_s")
    fig = px.line(m.to_pandas(), x="batch", y="MB_s", color="provider", line_dash="kind",
                  markers=True, log_x=True, facet_col="size",
                  category_orders={"provider": ORDER, "size": SIZES},
                  labels={"MB_s": "MB/s", "batch": "batch size (docs, log)"},
                  title="Goodput (useful data) vs wire rate (bytes moved)")
    fig.show()

    missing = sorted(set(raw["provider"].unique().to_list()) - set(gd["provider"].cast(pl.String).unique().to_list()))
    if missing:
        print(f"  not plotted (no wire-bytes measurement and no calibration constant): {', '.join(missing)}")

    print(f"  {'client':9} {'best goodput':>13} {'wire at that point':>19} {'efficiency':>11}")
    print("  " + "-" * 58)
    for prov in gd["provider"].unique().to_list():
        b_ = gd.filter(pl.col("provider") == prov).sort("goodput", descending=True).head(1)
        print(f"  {str(prov):9} {b_['goodput'][0]:10.1f} MB/s {b_['wire'][0]:15.1f} MB/s "
              f"{b_['efficiency'][0]:10.0%}   ({b_['src'][0]})")
    print()
    print("  efficiency = useful bytes delivered per byte moved. The shortfall is what the")
    print("  protocol spends encoding a 768-float vector as decimal text.")


  not plotted (no wire-bytes measurement and no calibration constant): topk-sql
  client     best goodput  wire at that point  efficiency
  ----------------------------------------------------------
  topk-rs         54.7 MB/s            56.4 MB/s        97%   (constant)
  topk-py         41.3 MB/s            42.6 MB/s        97%   (constant)
  topk-es         20.1 MB/s            99.0 MB/s        20%   (measured)

  efficiency = useful bytes delivered per byte moved. The shortfall is what the
  protocol spends encoding a 768-float vector as decimal text.


## Where the overhead comes from

Live measurement — needs a near-empty request to establish each client's fixed floor, which the sweep does not issue. `RUN_LIVE` guards it: running this during a sweep contaminates both.

In [9]:
RUN_LIVE = False

if RUN_LIVE:
    import sys, time
    sys.path.insert(0, os.path.join(ROOT, "python"))
    import topk_bench as tb
    from topk_bench.providers.topk_es import TopKESProvider

    q = pl.read_parquet("/tmp/topk-bench/queries-100k.parquet")
    vec = [float(x) for x in q["dense"].head(1).to_list()[0]]
    es, nat = TopKESProvider(), tb.TopKProvider()
    coll = nat.client.collection("x-100k")

    def timed(fn, n=200):
        fn()
        t = []
        for _ in range(n):
            a = time.perf_counter_ns(); fn(); t.append((time.perf_counter_ns() - a) / 1e6)
        return sum(t) / len(t)

    rows = [
        ("topk-es", "floor (GET /)",   timed(lambda: es.client.info())),
        ("topk-es", "search k=10",     timed(lambda: es.client.search(
            index="x-100k", knn={"field": "dense_embedding", "query_vector": vec, "k": 10},
            size=10, source_includes=["text", "int_filter", "keyword_filter"]))),
        ("topk-py", "floor (count)",   timed(lambda: coll.count())),
        ("topk-py", "query k=10",      timed(lambda: nat.query("x-100k", vec, 10, None, None))),
    ]
    fl = pl.DataFrame({"provider": [r[0] for r in rows], "what": [r[1] for r in rows],
                       "ms": [r[2] for r in rows]})
    px.bar(fl.to_pandas(), x="what", y="ms", color="provider", barmode="group",
           title="Per-request floor vs full query", labels={"ms": "ms"}).show()
else:
    print("RUN_LIVE = False")

RUN_LIVE = False
